**Patched copy (2026-10-14).** Cells marked `# PATCH` fix: unit-class tickers (`/` to `-`), the `/Canada` region misread, an `Additions` sheet with per-row asset class (incl. `Crypto`), a sourced `manual_fund_figures.csv` fallback, dropping the partial price week, an `AI` theme, a leveraged-fund exclusion, generated descriptions and an `allocation_policy` table. Before a full rebuild, copy the `Additions` sheet from `ETF_List_Additions.xlsx` into `ETF List Data.xlsx` and put `manual_fund_figures.csv` next to the notebook.

## Robo Adivisor Database builder.

This is the builder for the UNION Robo advisor database.  using real data from y-finace to provide all the features for our site.


In [5]:
import pandas as pd
import numpy as np

# We will build the original data frame which eventially some features will be pulled into another dataframe called context
schema = {
    'ticker': 'object',
    'ticker_yahoo': 'object',
    'name': 'object',
    'nameLong': 'object',
    'FundFamily': 'object',
    'exchange': 'object',
    'market': 'object',
    'asset_class': 'object',
    'region': 'object',
    'ETF_confirmed': 'boolean',
    'currency': 'object',
    'aum_cad_m': 'float64',
    'aum_total_assets': 'float64',
    'avg_volume_30d': 'float64',
    'theme': 'object',
    'theme_ESG_score': 'float64',
    'income': 'boolean',
    'income_yield': 'float64',
    'income_yield_trailing': 'float64',
    'leveraged': 'boolean',
    'MER': 'float64',
    'volatility': 'float64',
    'risk_tier': 'object',
    'volatility_52_week_high': 'float64',
    'volatility_52_week_low': 'float64',
    'risk_maximum_drawdown': 'float64',
    'volatility_beta': 'float64',
    'volatility_sharpe_ratio': 'float64',
    'threeYearAverageReturn': 'float64',
    'trailing_div_yield': 'float64',
    'risk_holdings_count': 'float64',
    'longBusinessSummary': 'object',
}

ETF_Core_data = pd.DataFrame(columns=schema.keys()).astype(schema)
ETF_Core_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ticker                   0 non-null      object 
 1   ticker_yahoo             0 non-null      object 
 2   name                     0 non-null      object 
 3   nameLong                 0 non-null      object 
 4   FundFamily               0 non-null      object 
 5   exchange                 0 non-null      object 
 6   market                   0 non-null      object 
 7   asset_class              0 non-null      object 
 8   region                   0 non-null      object 
 9   ETF_confirmed            0 non-null      boolean
 10  currency                 0 non-null      object 
 11  aum_cad_m                0 non-null      float64
 12  aum_total_assets         0 non-null      float64
 13  avg_volume_30d           0 non-null      float64
 14  theme                    0 non-null   

In [ ]:
file_path = 'ETF List Data.xlsx'

# Clean labels for each sheet (defined BEFORE the loop)
sheet_to_class = {
    'BOND ETFs': 'Bond',
    'Equity ETF': 'Equity',
    'Commodity ETF': 'Commodity',
    'Real Estate ETF': 'Real Estate',
}
# PATCH: the 'Additions' sheet (funds missing from the original list) carries its own 'Asset Class' column


excel_file = pd.ExcelFile(file_path)
sheet_names = excel_file.sheet_names
print(f"Sheet names found: {sheet_names}")

all_etfs = []

for sheet in sheet_names:
    df_sheet = pd.read_excel(file_path, sheet_name=sheet)
    df_sheet.columns = [str(col).strip().lower() for col in df_sheet.columns]

    ticker_col = [col for col in df_sheet.columns if 'ticker' in col]
    name_col = [col for col in df_sheet.columns if 'name' in col or 'fund' in col]

    if ticker_col and name_col:
        t_col, n_col = ticker_col[0], name_col[0]

        temp_df = df_sheet[[t_col, n_col]].rename(columns={t_col: 'ticker', n_col: 'name'})
        # Sheet name -> clean asset class label (falls back to the raw sheet name)
        temp_df['asset_class'] = sheet_to_class.get(sheet.strip(), sheet.strip())
        if 'asset class' in df_sheet.columns:                        # PATCH: per-row asset class
            temp_df['asset_class'] = df_sheet['asset class'].values
        all_etfs.append(temp_df)
    else:
        print(f"Could not identify required columns in sheet: {sheet}. Columns found: {df_sheet.columns.tolist()}")

if all_etfs:
    combined_raw_data = pd.concat(all_etfs, ignore_index=True)
    combined_raw_data = combined_raw_data.dropna(subset=['ticker', 'name'])

    # Check for tickers that appear on more than one sheet BEFORE dropping them
    dupes = combined_raw_data[combined_raw_data.duplicated('ticker', keep=False)]
    if not dupes.empty:
        print(f"{dupes['ticker'].nunique()} tickers appear on multiple sheets:")
        display(dupes.sort_values('ticker'))

    combined_raw_data = combined_raw_data.drop_duplicates(subset=['ticker'])

    ETF_Core_data = pd.concat([ETF_Core_data, combined_raw_data], ignore_index=True)
    ETF_Core_data = ETF_Core_data.astype(schema)

    print(f"Successfully loaded {len(ETF_Core_data)} unique ETFs into ETF_Core_data.")
    print(ETF_Core_data['asset_class'].value_counts(dropna=False))
    display(ETF_Core_data.head())
else:
    print("No matching data could be extracted from the Excel sheets.")



# build the yahoo finace reference table

This is where we will find the stocks as the orginal CSV had a bunch of bad listings

In [ ]:
#converting the bloomberg to yahoo tickers
import json, os, time
import pandas as pd
import yfinance as yf

BBG_MAP = {
    'CN': ('.TO', 'TSX (Canada)'),
    'US': ('',    'US'),
    'LN': ('.L',  'LSE (London)'),
    'GR': ('.DE', 'Xetra (Germany)'),
    'IM': ('.MI', 'Borsa Italiana (Milan)'),
    'FP': ('.PA', 'Euronext Paris'),
    'NA': ('.AS', 'Euronext Amsterdam'),
    'SW': ('.SW', 'SIX Swiss'),
    'AU': ('.AX', 'ASX (Australia)'),
    'HK': ('.HK', 'HKEX'),
    'JP': ('.T',  'Tokyo'),
}

def parse_bloomberg_ticker(raw):
    if pd.isna(raw):
        return pd.Series([None, None])
    parts = str(raw).strip().split()
    symbol = parts[0].replace('/', '-')     # PATCH: HSUV/U CN -> HSUV-U.TO (Yahoo uses a dash)
    code = parts[-1].upper() if len(parts) > 1 else ''
    sfx, market = BBG_MAP.get(code, ('', 'Other / Unknown'))
    return pd.Series([symbol + sfx, market])

ETF_Core_data[['ticker_yahoo', 'market']] = ETF_Core_data['ticker'].apply(parse_bloomberg_ticker)
ETF_Core_data['ticker_yahoo_initial'] = ETF_Core_data['ticker_yahoo']   # audit trail

print(ETF_Core_data['market'].value_counts())
print(ETF_Core_data[['ticker', 'ticker_yahoo', 'market']].head(10))

Confirming the stocks are aviable.

In [8]:
#verifying the tickers with yfinance and include the fallbacks
FALLBACKS = {
    ".TO": (".NE", ".V", ".CN"),  ".NE": (".TO", ".V", ".CN"),
    ".V":  (".TO", ".NE", ".CN"), ".CN": (".TO", ".NE", ".V"),
    ".DE": (".MI", ".SW", ".L", ".PA", ".AS"),
    ".L":  (".DE", ".MI", ".SW", ".PA", ".AS"),
    ".MI": (".DE", ".SW", ".L", ".PA", ".AS"),
    ".SW": (".DE", ".MI", ".L", ".PA", ".AS"),
    ".PA": (".DE", ".MI", ".SW", ".L", ".AS"),
    ".AS": (".DE", ".MI", ".SW", ".L", ".PA"),
}

CACHE_FILE = 'yahoo_full_cache.json'      # new file: the old one only held 9 fields
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def fetch(t):
    try:
        info = yf.Ticker(t).info
        if info and (info.get('longName') or info.get('shortName')):
            return info
    except Exception:
        pass
    return None

def verify(initial):
    sfx = '.' + initial.rsplit('.', 1)[1] if '.' in initial else ''
    base = initial.rsplit('.', 1)[0] if sfx else initial
    for cand in [initial] + [base + s for s in FALLBACKS.get(sfx, ())]:
        info = fetch(cand)
        time.sleep(0.3)
        if info:
            return cand, info          # full info dict
    return None, None

FULL_RUN = True  # set to False for testing with a smaller sample
scope = ETF_Core_data.index if FULL_RUN else ETF_Core_data.groupby('asset_class').head(25).index

for n, idx in enumerate(scope, 1):
    bbg = ETF_Core_data.at[idx, 'ticker']
    if bbg not in cache:
        resolved, info = verify(ETF_Core_data.at[idx, 'ticker_yahoo_initial'])
        cache[bbg] = {'resolved': resolved, 'data': info}
    if n % 25 == 0:
        json.dump(cache, open(CACHE_FILE, 'w'), default=str)
        print(f"{n} / {len(scope)} checked...")
json.dump(cache, open(CACHE_FILE, 'w'), default=str)

25 / 3110 checked...
50 / 3110 checked...
75 / 3110 checked...
100 / 3110 checked...
125 / 3110 checked...
150 / 3110 checked...
175 / 3110 checked...
200 / 3110 checked...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ZAAA.TO"}}}


225 / 3110 checked...


HTTP Error 404: {"finance":{"result":null,"error":{"code":"Not Found","description":"HTTP 404 Not Found"}}}
HTTP Error 404: {"finance":{"result":null,"error":{"code":"Not Found","description":"HTTP 404 Not Found"}}}
HTTP Error 404: {"finance":{"result":null,"error":{"code":"Not Found","description":"HTTP 404 Not Found"}}}
HTTP Error 404: {"finance":{"result":null,"error":{"code":"Not Found","description":"HTTP 404 Not Found"}}}


250 / 3110 checked...


...

3025 / 3110 checked...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OIL.DE"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OIL.MI"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OIL.PA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OIL.AS"}}}


3050 / 3110 checked...
3075 / 3110 checked...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SREEEX.MI"}}}


3100 / 3110 checked...


In [9]:
#we will remove the dropped tickers but keep a record of the ones we dropped for future reference

# Find ETFs that were checked and not found, using the cache
checked = ETF_Core_data['ticker'].map(lambda t: t in cache)
unresolved = ETF_Core_data['ticker'].map(lambda t: t in cache and cache[t]['resolved'] is None)

nf = ETF_Core_data[checked & unresolved]
print(len(nf), "not found")
display(nf[['ticker', 'ticker_yahoo_initial', 'name', 'asset_class']])

nf.to_csv('etfs_not_found_on_yahoo.csv', index=False)

# Drop them WITHOUT reset_index so the row numbers in `scope` stay valid
ETF_Core_data = ETF_Core_data.drop(index=nf.index)
scope = scope.intersection(ETF_Core_data.index)
print(len(scope), "ETFs remain in the test scope")

94 not found


,ticker,ticker_yahoo_initial,name,asset_class
127,JGBUX I2,JGBUX,iShares Japan Govt Bond UCITS ETF,Bond
187,HSUV/U CN,HSUV/U.TO,Global X USD Cash Maximizer Corporate Class ETF,Bond
233,XFLI/U CN,XFLI/U.TO,iShares Flexible Monthly Income ETF,Bond
299,ETFLCD GR,ETFLCD.DE,Deka iBoxx EUR Liquid Corporates Diversified U...,Bond
345,MNU/U CN,MNU/U.TO,Purpose USD Cash Management Fund,Bond
...,...,...,...,...
2729,APSDEX GR,APSDEX.DE,iShares Dow Jones Asia Pacific Select Dividend...,Equity
2847,ETFSV2P GR,ETFSV2P.DE,Deka STOXX Europe Strong Value 20 UCITS ETF,Equity
2935,EWG2 GR,EWG2.DE,EUWAX GOLD II,Commodity
3016,DJCOMEX GR,DJCOMEX.DE,iShares Diversified Commodity Swap UCITS ETF DE,Commodity


3016 ETFs remain in the test scope


2B map Yahoo fields into your columns

In [ ]:
def first(d, *keys):
    for k in keys:
        if d.get(k) is not None:
            return d[k]
    return None

for idx in scope:
    c = cache[ETF_Core_data.at[idx, 'ticker']]
    initial = ETF_Core_data.at[idx, 'ticker_yahoo_initial']
    if c['resolved'] is None:
        ETF_Core_data.at[idx, 'verify_status'] = 'not_found'
        continue
    d = c['data']
    ETF_Core_data.at[idx, 'ticker_yahoo']  = c['resolved']
    ETF_Core_data.at[idx, 'verify_status'] = 'confirmed' if c['resolved'] == initial else 'rescued'
    ETF_Core_data.at[idx, 'nameLong']      = first(d, 'longName', 'shortName')
    ETF_Core_data.at[idx, 'FundFamily']    = d.get('fundFamily')
    ETF_Core_data.at[idx, 'exchange']      = d.get('exchange')
    ETF_Core_data.at[idx, 'currency']      = d.get('currency')
    ETF_Core_data.at[idx, 'ETF_confirmed'] = (d.get('quoteType') == 'ETF')
    ETF_Core_data.at[idx, 'aum_total_assets'] = first(d, 'totalAssets', 'netAssets')   # PATCH: netAssets fallback
    ETF_Core_data.at[idx, 'avg_volume_30d']   = first(d, 'averageVolume', 'averageVolume10days')
    ETF_Core_data.at[idx, 'income_yield']          = d.get('yield')
    ETF_Core_data.at[idx, 'income_yield_trailing'] = d.get('trailingAnnualDividendYield')
    ETF_Core_data.at[idx, 'volatility_52_week_high'] = d.get('fiftyTwoWeekHigh')
    ETF_Core_data.at[idx, 'volatility_52_week_low']  = d.get('fiftyTwoWeekLow')
    ETF_Core_data.at[idx, 'volatility_beta'] = first(d, 'beta3Year', 'beta')
    ETF_Core_data.at[idx, 'threeYearAverageReturn'] = d.get('threeYearAverageReturn')
    ETF_Core_data.at[idx, 'MER'] = first(d, 'netExpenseRatio', 'annualReportExpenseRatio')
    ETF_Core_data.at[idx, 'longBusinessSummary'] = d.get('longBusinessSummary')
    ETF_Core_data.at[idx, 'morningstar_category'] = d.get('category')   # add to schema if you keep it

ETF_Core_data['trailing_div_yield'] = ETF_Core_data['income_yield_trailing']
print(ETF_Core_data['verify_status'].value_counts(dropna=False))

2c: volatility, drawdown, Sharpe from price history

In [11]:
RF = 0.0   # annual risk-free rate for Sharpe; set to e.g. 0.04 if you want one
tickers = ETF_Core_data.loc[scope, 'ticker_yahoo'].dropna().unique().tolist()
stats = {}

for i in range(0, len(tickers), 50):
    batch = tickers[i:i+50]
    px = yf.download(batch, period='3y', auto_adjust=True, progress=False)['Close']
    if isinstance(px, pd.Series):
        px = px.to_frame(batch[0])
    for t in px.columns:
        s = px[t].dropna()
        if len(s) < 200:
            continue                                   # not enough history
        r = s.pct_change().dropna()
        vol = r.std() * np.sqrt(252)
        stats[t] = {
            'volatility': vol,
            'risk_maximum_drawdown': (s / s.cummax() - 1).min(),
            'volatility_sharpe_ratio': (r.mean() * 252 - RF) / vol if vol else np.nan,
        }
    time.sleep(1)

stats_df = pd.DataFrame(stats).T
for col in stats_df.columns:
    ETF_Core_data[col] = (
        ETF_Core_data['ticker_yahoo']
        .map(stats_df[col])
        .combine_first(ETF_Core_data[col])
    )

$FCIG.TO: possibly delisted; no price data found  (period=3y)

1 Failed download:
['FCIG.TO']: possibly delisted; no price data found  (period=3y)
$XB13.SW: No data found, symbol may be delisted

1 Failed download:
['XB13.SW']: No data found, symbol may be delisted
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
$SMIEX.DE: No data found, symbol may be delisted

1 Failed download:
['SMIEX.DE']: No data found, symbol may be delisted
$MMSD.TO: possibly delisted; no price data found  (period=3y)

1 Failed download:
['MMSD.TO']: possibly delisted; no price data found  (period=3y)
$GBSE.DE: No data found, symbol may be delisted

1 Failed download:
['GBSE.DE']: No data found, symbol may be delisted
$SREEEX.SW: possibly delisted; no price data found  (period=3y)

1 

2d derived columns

In [12]:
# income: pays a distribution
y = ETF_Core_data['income_yield_trailing'].fillna(ETF_Core_data['income_yield'])
ETF_Core_data['income'] = (y > 0).where(y.notna())

# leveraged / inverse: name-based
lev = r'(?:\b[23]x\b|(?<!\w)-[123]x\b|\bleveraged\b|\binverse\b|\bbull\b|\bbear\b|\bultrashort\b|\bultrapro\b|\bdaily\b.*\b[23]x\b)'
text = ETF_Core_data['name'].fillna('') + ' ' + ETF_Core_data['nameLong'].fillna('')
ETF_Core_data['leveraged'] = text.str.lower().str.contains(lev, regex=True)

# AUM in CAD (millions)
need = ETF_Core_data['currency'].dropna().unique()
rate = {'CAD': 1.0}
for cur in need:
    if cur not in rate and cur != 'GBp':
        try:
            rate[cur] = yf.Ticker(f'{cur}CAD=X').history(period='5d')['Close'].iloc[-1]
        except Exception:
            pass
ETF_Core_data['aum_cad_m'] = (ETF_Core_data['aum_total_assets']
                              * ETF_Core_data['currency'].map(rate) / 1e6)

display(ETF_Core_data[ETF_Core_data['leveraged']][['ticker', 'name', 'nameLong']])
print(ETF_Core_data['leveraged'].value_counts())

,ticker,name,nameLong
20,ERNE LN,iShares EUR Ultrashort Bond UCITS ETF,iShares € Ultrashort Bond UCITS ETF EUR (Dist)
49,XT01 GR,Xtrackers US Treasuries Ultrashort Bond UCITS ETF,Xtrackers US Treasuries Ultrashort Bond UCITS ...
65,ERND LN,iShares USD Ultrashort Bond UCITS ETF,iShares $ Ultrashort Bond UCITS ETF USD (Dist)
72,ERNS LN,iShares GBP Ultrashort Bond UCITS ETF,iShares £ Ultrashort Bond UCITS ETF GBP (Dist)
106,EUED GR,iShares EUR UltraShort Bond ESG SRI UCITS ETF,iShares € Ultrashort Bond ESG SRI UCITS ETF EU...
280,UESD LN,iShares GBP UltraShort Bond ESG SRI UCITS ETF,iShares £ Ultrashort Bond ESG SRI UCITS ETF GB...
717,TUCP GR,Janus Henderson EUR Ultrashort IG Bond Paris-A...,Janus Henderson EUR Ultrashort IG Bond Paris-a...
2113,SC2X GR,Scalable MSCI AC World Leveraged Daily Swap Xt...,Scalable MSCI AC World Leveraged Daily Swap Xt...
2615,SPCH LN,GraniteShares 3x Long SpaceX Daily ETP,GraniteShares 3x Long SpaceX Daily ETP
3033,USLV US,Direxion Daily Silver Bull 2X ETF,Direxion Daily Silver Bull 2X ETF


leveraged
False    3005
True       11
Name: count, dtype: int64


In [13]:
print(ETF_Core_data['aum_cad_m'].isna().sum(), "rows without aum_cad_m")
print(ETF_Core_data[ETF_Core_data['aum_cad_m'].isna()]['currency'].value_counts(dropna=False))

717 rows without aum_cad_m
currency
USD     250
EUR     179
CAD     128
GBp     126
GBP      30
None      3
CHF       1
Name: count, dtype: int64


# check point save what we have!

In [ ]:
#save the dataframe to a pickle file for later use
ETF_Core_data.to_pickle('universe_checkpoint.pkl')

# now start cutting down to what we want Exclusions

In [ ]:
import os
import numpy as np
df = ETF_Core_data

# Listing country from the resolved Yahoo suffix
sfx = df['ticker_yahoo'].str.extract(r'\.(\w+)$')[0]
df['listing_country'] = np.select(
    [df['ticker_yahoo'].isna(), sfx.isna(), sfx.isin(['TO', 'NE', 'V', 'CN']), sfx.eq('L'), sfx.eq('DE')],
    ['Unknown', 'US', 'Canada', 'UK', 'Germany'], default='Other')

# Inverse funds, by name
text = (df['name'].fillna('') + ' ' + df['nameLong'].fillna('')).str.lower()
df['is_inverse'] = text.str.contains(
    r'(?:\binverse\b|\bbear\b|ultrashort|(?<!\w)-[123]x\b|\bshort\s+(?:s&p|nasdaq|dow|russell|qqq|msci|ftse))',
    regex=True)

# PATCH: fill fees / fund sizes Yahoo lacks from a sourced, dated file (Yahoo value always wins)
if os.path.exists('manual_fund_figures.csv'):
    mf = pd.read_csv('manual_fund_figures.csv').set_index('ticker')
    df['MER'] = df['MER'].fillna(df['ticker'].map(mf['mer_pct']))
    df['aum_total_assets'] = df['aum_total_assets'].fillna(df['ticker'].map(mf['aum_total_assets']))
    fx = df['currency'].map(rate).fillna(1.0)          # 'rate' comes from the AUM-in-CAD cell
    df['aum_cad_m'] = df['aum_cad_m'].fillna(df['aum_total_assets'] * fx / 1e6)

# Check fill rates BEFORE deciding which figures are required
REQUIRED = ['aum_total_assets', 'avg_volume_30d', 'MER']
print(df[REQUIRED].notna().mean().round(2))

size_ok = np.where(df['listing_country'].eq('Canada'),
                   df['aum_cad_m'] >= 50,                     # PATCH: C$50M for Canadian-listed (was C$100M; team decision 2026-10-24)
                   df['aum_total_assets'] / 1e6 >= 250)       # US$250M for US-listed
vol_ok = df['avg_volume_30d'] >= np.where(df['listing_country'].eq('Canada'), 1_000, 10_000)

rules = [
    ('not a Canadian or US listing', ~df['listing_country'].isin(['Canada', 'US'])),
    ('not confirmed as an ETF',      ~df['ETF_confirmed'].fillna(False).astype(bool)),
    ('inverse fund',                 df['is_inverse']),
    ('missing key figures',          df[REQUIRED].isna().any(axis=1)),
    ('below size minimum',           ~pd.Series(size_ok, index=df.index)),
    ('below volume minimum',         ~vol_ok),
]

df['exclusion_reason'] = pd.NA
print(f"\nStart: {len(df)}")
for reason, fail in rules:
    hit = df['exclusion_reason'].isna() & fail.fillna(False)
    df.loc[hit, 'exclusion_reason'] = reason
    print(f"{reason:32s} removes {hit.sum():5d} -> {df['exclusion_reason'].isna().sum():5d} remain")

df['eligible'] = df['exclusion_reason'].isna()

Build ETF PRICES Database for sqliite

In [ ]:
import sqlite3, datetime as dt

con = sqlite3.connect('etf_prices.db')
con.executescript("""
CREATE TABLE IF NOT EXISTS price_weekly (
    ticker TEXT NOT NULL, week_end TEXT NOT NULL, adj_close REAL NOT NULL,
    PRIMARY KEY (ticker, week_end)) WITHOUT ROWID;
CREATE TABLE IF NOT EXISTS price_meta (
    ticker TEXT PRIMARY KEY, ticker_yahoo TEXT, currency TEXT, status TEXT,
    first_week TEXT, last_week TEXT, n_weeks INTEGER, fetched_at TEXT);
""")

todo = ETF_Core_data.loc[ETF_Core_data['eligible'], ['ticker', 'ticker_yahoo', 'currency']]
done = {r[0] for r in con.execute("SELECT ticker FROM price_meta")}
todo = todo[~todo['ticker'].isin(done)].reset_index(drop=True)
print(len(todo), "funds to pull")

def download(syms, retries=3):
    for attempt in range(retries):
        try:
            px = yf.download(syms, period='3y', interval='1d', auto_adjust=True,
                             progress=False, threads=False)['Close']
            if px is not None and not px.empty:
                return px
        except Exception:
            pass
        time.sleep(10 * (attempt + 1))
    return None

BATCH = 20
for i in range(0, len(todo), BATCH):
    chunk = todo.iloc[i:i + BATCH]
    syms = chunk['ticker_yahoo'].tolist()
    px = download(syms)
    if px is None:
        print(f"Batch {i}: no response, likely rate-limited. Pausing 60s; these retry on the next run.")
        time.sleep(60)
        continue                                  # nothing recorded, so it is retried later
    if isinstance(px, pd.Series):
        px = px.to_frame(syms[0])
    wk = px.resample('W-FRI').last()              # one consistent week-ending date for all exchanges
    wk = wk[wk.index <= pd.Timestamp.today().normalize()]   # PATCH: drop the current partial week
    now = dt.datetime.now().isoformat(timespec='seconds')

    for _, r in chunk.iterrows():
        s = wk[r['ticker_yahoo']].dropna() if r['ticker_yahoo'] in wk.columns else pd.Series(dtype=float)
        if s.empty:
            con.execute("INSERT OR REPLACE INTO price_meta VALUES (?,?,?,?,?,?,?,?)",
                        (r['ticker'], r['ticker_yahoo'], r['currency'], 'no_data', None, None, 0, now))
            continue
        con.executemany("INSERT OR REPLACE INTO price_weekly VALUES (?,?,?)",
                        [(r['ticker'], d.strftime('%Y-%m-%d'), float(v)) for d, v in s.items()])
        con.execute("INSERT OR REPLACE INTO price_meta VALUES (?,?,?,?,?,?,?,?)",
                    (r['ticker'], r['ticker_yahoo'], r['currency'], 'ok',
                     s.index[0].strftime('%Y-%m-%d'), s.index[-1].strftime('%Y-%m-%d'), len(s), now))
    con.commit()
    print(f"{min(i + BATCH, len(todo))} / {len(todo)} done")
    time.sleep(3)

print(pd.read_sql("SELECT status, COUNT(*) n, ROUND(AVG(n_weeks)) avg_weeks FROM price_meta GROUP BY status", con))
con.close()

In [17]:
#check the missed rescued listings that came back no_data

meta = pd.read_sql("SELECT ticker, status, n_weeks FROM price_meta", sqlite3.connect('etf_prices.db'))
chk = ETF_Core_data.merge(meta, on='ticker')
display(chk[(chk['verify_status'] == 'rescued') & (chk['status'] == 'no_data')][['ticker', 'ticker_yahoo_initial', 'ticker_yahoo', 'nameLong']])

,ticker,ticker_yahoo_initial,ticker_yahoo,nameLong


# Volatility, drawdown, Sharpe and risk tier from

In [18]:
import sqlite3
import numpy as np
import pandas as pd

con = sqlite3.connect('etf_prices.db')
px = pd.read_sql("SELECT ticker, week_end, adj_close FROM price_weekly", con, parse_dates=['week_end'])
meta = pd.read_sql("SELECT ticker, status, n_weeks FROM price_meta", con)
con.close()

# Eligible funds that still have no price pull (rate-limit leftovers): rerun the pull cell if > 0
not_pulled = ETF_Core_data['eligible'] & ~ETF_Core_data['ticker'].isin(meta['ticker'])
print(not_pulled.sum(), "eligible funds have no price pull yet")

# One column per fund, one row per week; drop the current partial week
wide = px.pivot(index='week_end', columns='ticker', values='adj_close').sort_index()
wide = wide[wide.index <= pd.Timestamp.today().normalize()]

ret = wide.pct_change(fill_method=None)       # weekly total returns (prices are dividend-adjusted)
RF = 0.0                                      # annual risk-free rate for Sharpe; set e.g. 0.03 if wanted

vol = ret.std() * np.sqrt(52)                 # annualized from weekly returns, as the spec says
stats = pd.DataFrame({
    'volatility': vol,
    'risk_maximum_drawdown': (wide / wide.cummax() - 1).min(),
    'volatility_sharpe_ratio': (ret.mean() * 52 - RF) / vol,
})
for col in stats.columns:
    ETF_Core_data[col] = ETF_Core_data['ticker'].map(stats[col])
ETF_Core_data['price_weeks'] = ETF_Core_data['ticker'].map(meta.set_index('ticker')['n_weeks'])

# Risk tier: Low 0-11%, Medium 11-20%, High 20%+ annual volatility
ETF_Core_data['risk_tier'] = pd.cut(
    ETF_Core_data['volatility'], bins=[0, 0.11, 0.20, np.inf],
    labels=['Low', 'Medium', 'High'], right=False).astype('object')

# Last exclusion: about 3 years of history (150 weeks allows for holiday gaps)
MIN_WEEKS = 150
short = ETF_Core_data['eligible'] & ~(ETF_Core_data['price_weeks'] >= MIN_WEEKS)
ETF_Core_data.loc[short, 'exclusion_reason'] = 'under 3 years of prices (or none)'
ETF_Core_data['eligible'] = ETF_Core_data['exclusion_reason'].isna()

print(f"\nEligible after history check: {ETF_Core_data['eligible'].sum()}")
print(ETF_Core_data.loc[ETF_Core_data['eligible'], 'risk_tier'].value_counts(dropna=False))
print(pd.crosstab(ETF_Core_data.loc[ETF_Core_data['eligible'], 'asset_class'],
                  ETF_Core_data.loc[ETF_Core_data['eligible'], 'risk_tier']))

0 eligible funds have no price pull yet

Eligible after history check: 575
risk_tier
Medium    403
Low       113
High       59
Name: count, dtype: int64
risk_tier    High  Low  Medium
asset_class                   
Bond            0   88       1
Commodity      18    1      29
Equity         39   24     345
Real Estate     2    0      28


In [19]:
#sanity check

elig = ETF_Core_data[ETF_Core_data['eligible']]
print(elig.groupby('asset_class')['volatility'].describe().round(3))

# Likely data problems: stale prices (near 0) or splits/bad ticks (very high)
odd = elig[(elig['volatility'] < 0.005) | (elig['volatility'] > 0.60)]
display(odd[['ticker', 'ticker_yahoo', 'name', 'volatility', 'price_weeks']])


             count   mean    std    min    25%    50%    75%    max
asset_class                                                        
Bond          89.0  0.048  0.022  0.002  0.038  0.049  0.061  0.138
Commodity     48.0  0.250  0.182  0.081  0.143  0.186  0.282  1.074
Equity       408.0  0.155  0.039  0.055  0.134  0.146  0.169  0.358
Real Estate   30.0  0.160  0.017  0.140  0.148  0.158  0.163  0.202


,ticker,ticker_yahoo,name,volatility,price_weeks
3,PULS US,PULS,PGIM Ultra Short Bond ETF,0.004087,158.0
4,MINT US,MINT,PIMCO Enhanced Short Maturity Active Exchange-...,0.002699,158.0
40,CASH CN,CASH.TO,Global X High Interest Savings ETF,0.001977,157.0
110,RAVI US,RAVI,Northern Trust Ultra-Short Fixed Income ETF,0.004021,158.0
2969,AGQ US,AGQ,ProShares Ultra Silver,0.815451,158.0
3019,BOIL US,BOIL,ProShares Ultra Bloomberg Natural Gas,1.074115,158.0


In [20]:
# save another checkpoint for the files

from datetime import datetime

def checkpoint(label):
    fn = f"checkpoint_{datetime.now():%Y%m%d_%H%M}_{label}.pkl"
    ETF_Core_data.to_pickle(fn)
    print("saved", fn)

checkpoint('risk_tier')

saved checkpoint_20261013_0025_risk_tier.pkl


# region, ESG, theme, income and hedging

In [ ]:
import re
import numpy as np
import pandas as pd

ac = ETF_Core_data['asset_class']
cat = ETF_Core_data['morningstar_category'].fillna('') if 'morningstar_category' in ETF_Core_data.columns else ''
raw = (ETF_Core_data['name'].fillna('') + ' | ' + ETF_Core_data['nameLong'].fillna('') + ' | ' + cat).str.lower()

ETF_Core_data['cad_hedged'] = raw.str.contains(r'(?<!un)hedged', regex=True)

# Cleaned text for matching: drop brand and currency words that look like regions
txt = (raw.str.replace(r'global\s*x', ' ', regex=True)
          .str.replace(r'canada life', ' ', regex=True)
          .str.replace(r'/\s*canada\b|\betf canada\b', ' ', regex=True)   # PATCH: Canadian share class, not a region
          .str.replace(r'cad[- ]?hedged|hedged to cad|\bhedged\b|\(cad\)|\(usd\)|\bcad\b|\busd\b|u\.?s\.? dollar|canadian dollar',
                       ' ', regex=True))

# ---------- region ----------
REGION_RULES = [   # first match wins
    ('Emerging Markets', r'emerging|\bem\b|\bchina\b|\bchinese\b|\bindia\b|latin america|\bbrazil\b|\btaiwan\b|\bkorea\b|frontier|\bbric|\bvietnam\b|\bindonesia\b|\bmexico\b|south africa|\bturkey\b|\bsaudi\b|\bthailand\b|asia ex[- ]?japan'),
    ('International',    r'\bex[- ]?u\.?s\b|\bex[- ]?north america|\beafe\b|international|\bintl\b|foreign|europe|eurozone|\beuro\b|japan|nikkei|topix|pacific|australia|\buk\b|united kingdom|germany|france|switzerland|nordic|\bdax\b|stoxx|\bgilts?\b|\bbunds?\b|sterling|developed ex'),
    ('Global',           r'global|world|all[- ]country|\bacwi\b|worldwide|\bex[- ]?canada'),
    ('North America',    r'north america'),
    ('Canada',           r'canad|\btsx\b|\bcdn\b|maple'),
    ('US',               r'\bu\.?s\.?a?\b|united states|s&p ?500|nasdaq|russell|dow jones|\btreasur|t-bill|\btips\b'),
]

def derive_region(t, asset_class, listing):
    for region, pat in REGION_RULES:
        if re.search(pat, t):
            return region, 'name'
    if asset_class == 'Commodity':
        return 'Global', 'commodity_default'
    if asset_class == 'Crypto':                        # PATCH: crypto has no home country
        return 'Global', 'crypto_default'
    if listing == 'US':
        return 'US', 'listing_default'
    if listing == 'Canada' and asset_class in ('Bond', 'Real Estate'):
        return 'Canada', 'listing_default'
    if listing == 'Canada' and asset_class == 'Equity':
        return 'North America', 'listing_default'     # Canadian-listed stock fund, no region word
    return 'Unknown', 'none'

res = [derive_region(t, a, l) for t, a, l in zip(txt, ac, ETF_Core_data['listing_country'])]
ETF_Core_data['region'] = [r[0] for r in res]
ETF_Core_data['region_source'] = [r[1] for r in res]

REGION_OVERRIDE = {   # Bloomberg ticker -> region, for funds the rules can't read
    # 'PMIF CN': 'Global',
    'SMVP CN': 'US',      # PATCH: Hamilton Champions US Dividend, read as Canada
}
for t, v in REGION_OVERRIDE.items():
    ETF_Core_data.loc[ETF_Core_data['ticker'] == t, ['region', 'region_source']] = [v, 'manual']

# ---------- ESG flag ----------
ESG_KEYWORDS = [
    r'\besg\b', r'sustainab', r'\bsri\b', r'socially responsible', r'\bresponsible\b',
    r'climate', r'low[- ]carbon', r'\bcarbon\b', r'fossil[- ](?:fuel[- ])?free',
    r'paris[- ]aligned', r'green bond', r'\bimpact\b', r'net[- ]zero', r'\bsdg',
]
ETF_Core_data['is_esg'] = txt.str.contains('|'.join(ESG_KEYWORDS), regex=True)

ESG_OVERRIDE = []   # Bloomberg tickers to force to ESG after you've read their summaries
ETF_Core_data.loc[ETF_Core_data['ticker'].isin(ESG_OVERRIDE), 'is_esg'] = True

# ---------- theme (one label; ESG only if nothing else fits) ----------
THEME_RULES = [
    ('Crypto',         r'bitcoin|\bbtc\b|\bether\b|ethereum|crypto|blockchain|digital asset'),
    ('Clean Energy',   r'clean energy|renewable|\bsolar\b|\bwind\b|hydrogen|green energy|clean ?tech|energy transition|\bbattery\b|lithium'),
    ('AI',             r'artificial intelligence|\bai\b|generative|robotic'),   # PATCH: own theme for the AI add-on
    ('Technology',     r'technolog|\btech\b|semiconductor|software|\bcloud\b|cyber|artificial intelligence|\bai\b|robotics|\binternet\b|fintech|innovation|\bdigital\b|e-?commerce'),
    ('Health Care',    r'health|biotech|pharma|genomic|medical|life sciences'),
    ('Financials',     r'financial|\bbanks?\b|insurance|insurers'),
    ('Infrastructure', r'infrastructure|utilit'),
    ('Resources',      r'\bgold\b|\bsilver\b|platinum|palladium|precious|\bmetals?\b|mining|miners|materials|resource|\boil\b|\bgas\b|petroleum|\benergy\b|pipeline|uranium|copper|agricultur|timber'),
    ('Consumer',       r'consumer|staples|discretionary'),
    ('Industrials',    r'\bindustrials?\b(?! average)'),
]
INFL = r'inflation|real return|\btips\b'

def derive_theme(t, asset_class, esg):
    if asset_class == 'Bond':
        if re.search(INFL, t):
            return 'Inflation-Protected'
    else:
        for name, pat in THEME_RULES:
            if re.search(pat, t):
                return name
    return 'ESG' if esg else None

ETF_Core_data['theme'] = [derive_theme(t, a, e) for t, a, e in zip(txt, ac, ETF_Core_data['is_esg'])]

# ---------- income: dividend-focused (stock-type funds only) ----------
INCOME = r'dividend|\bincome\b|distribution|covered call|buy-?write|\byield\b|aristocrat|payout'
ETF_Core_data['income'] = (txt.str.contains(INCOME, regex=True) & ac.isin(['Equity', 'Real Estate'])).astype('boolean')

#etf_category, the spec’s seven core classes plus the others

In [ ]:
reg = ETF_Core_data['region']
theme = ETF_Core_data['theme'].fillna('')

CASH = r'high interest|savings|\bcash\b|money market|t-bill|treasury bill|ultra[- ]?short|short maturity|overnight'
SHORT_TERM = r'short[- ]?term|short[- ]?duration|short[- ]?dated|short government|\b[01]\s?(?:-|–|to)\s?[35]\s?(?:yr|year)|\b1\s?(?:-|–|to)\s?5\b'

is_cash = (ac == 'Bond') & txt.str.contains(CASH, regex=True)
short_term = txt.str.contains(SHORT_TERM, regex=True)
sector_theme = ac.isin(['Equity', 'Bond']) & (theme != '') & (theme != 'ESG')

cond = [
    ac == 'Crypto',                                   # PATCH: crypto -> Sector or theme (theme 'Crypto')
    ac == 'Real Estate',
    ac == 'Commodity',
    is_cash,
    sector_theme,
    (ac == 'Equity') & (reg == 'Canada'),
    (ac == 'Equity') & (reg == 'US'),
    (ac == 'Equity') & (reg == 'International'),
    (ac == 'Equity') & (reg == 'Emerging Markets'),
    (ac == 'Bond') & (reg == 'Canada') & short_term,
    (ac == 'Bond') & (reg == 'Canada'),
    (ac == 'Bond') & reg.isin(['US', 'International', 'Emerging Markets', 'Global']),
]
choice = ['Sector or theme', 'Real estate', 'Commodities', 'Cash', 'Sector or theme',
          'Canadian stocks', 'US stocks', 'International stocks', 'Emerging markets stocks',
          'Canadian short-term bonds', 'Canadian bonds (all maturities)', 'Global bonds (incl. US)']
ETF_Core_data['etf_category'] = np.select(cond, choice, default='Other')

CATEGORY_OVERRIDE = {   # Bloomberg ticker -> etf_category
    # 'PMIF CN': 'Global bonds (incl. US)',
}
for t, v in CATEGORY_OVERRIDE.items():
    ETF_Core_data.loc[ETF_Core_data['ticker'] == t, 'etf_category'] = v

In [23]:
#reviewer
elig = ETF_Core_data[ETF_Core_data['eligible']]
print(len(elig), "eligible funds\n")
print(pd.crosstab(elig['etf_category'], elig['region'], margins=True), "\n")
print(elig['region_source'].value_counts(), "\n")
print(elig['theme'].value_counts(dropna=False), "\n")
print("ESG:", int(elig['is_esg'].sum()), " Income:", int(elig['income'].sum()), "\n")
print("Median trailing yield by income flag (should be higher for True):")
print(elig.groupby('income')['income_yield_trailing'].median())

cols = ['ticker', 'name', 'asset_class', 'listing_country', 'region', 'etf_category', 'theme']
display(elig[(elig['region'] == 'Unknown') | (elig['etf_category'] == 'Other')][cols])

575 eligible funds

region                           Canada  Emerging Markets  Global  \
etf_category                                                        
Canadian bonds (all maturities)      14                 0       0   
Canadian stocks                       7                 0       0   
Cash                                  1                 0       0   
Commodities                           0                 0      43   
Emerging markets stocks               0                88       0   
Global bonds (incl. US)               0                14      14   
International stocks                  0                 0       0   
Other                                 0                 0      35   
Real estate                           5                 0       7   
Sector or theme                       0                 0       2   
US stocks                             0                 0       0   
All                                  27               102     101   

region       

,ticker,name,asset_class,listing_country,region,etf_category,theme
818,VT US,Vanguard Total World Stock ETF,Equity,US,Global,Other,None
828,ACWI US,iShares MSCI ACWI ETF,Equity,US,Global,Other,None
858,XEQT CN,iShares Core Equity ETF Portfolio,Equity,Canada,Unknown,Other,None
872,VEQT CN,Vanguard All-Equity ETF Portfolio,Equity,Canada,Unknown,Other,None
902,IOO US,iShares Global 100 ETF,Equity,US,Global,Other,None
919,URTH US,iShares MSCI World ETF,Equity,US,Global,Other,None
945,JGLO US,JPMorgan Global Select Equity ETF,Equity,US,Global,Other,None
1015,ACWV US,iShares MSCI Global Min Vol Factor ETF,Equity,US,Global,Other,None
1045,VXC CN,Vanguard FTSE Global All Cap ex Canada Index ETF,Equity,Canada,Global,Other,None
1100,SPGM US,State Street SPDR Portfolio MSCI Global Stock ...,Equity,US,Global,Other,None


In [25]:
from datetime import datetime

def checkpoint(label):
    fn = f"checkpoint_{datetime.now():%Y%m%d_%H%M}_{label}.pkl"
    ETF_Core_data.to_pickle(fn)
    print("saved", fn)

checkpoint('categories')

saved checkpoint_20261013_0052_categories.pkl


## final check

In [ ]:
import json, os, shutil, sqlite3
from datetime import datetime
import numpy as np
import pandas as pd

ETF_Core_data.to_pickle(f"checkpoint_{datetime.now():%Y%m%d_%H%M}_pre_export.pkl")

# 1) MER in one unit (percent: 0.09 means 0.09%), rebuilt from the cached Yahoo data
cache = json.load(open('yahoo_full_cache.json'))

def mer_percent(bbg):
    d = (cache.get(bbg) or {}).get('data') or {}
    if d.get('netExpenseRatio') is not None:
        return d['netExpenseRatio']                    # Yahoo reports this in percent
    if d.get('annualReportExpenseRatio') is not None:
        return d['annualReportExpenseRatio'] * 100    # Yahoo reports this as a fraction
    return np.nan

ETF_Core_data['mer_pct'] = ETF_Core_data['ticker'].map(mer_percent)
if os.path.exists('manual_fund_figures.csv'):         # PATCH: sourced manual fees where Yahoo has none
    mf = pd.read_csv('manual_fund_figures.csv').set_index('ticker')
    ETF_Core_data['mer_pct'] = ETF_Core_data['mer_pct'].fillna(ETF_Core_data['ticker'].map(mf['mer_pct']))

# 2) Leverage factor (2 or 3) from the name; leveraged = factor > 1
lt = (ETF_Core_data['name'].fillna('') + ' ' + ETF_Core_data['nameLong'].fillna('')).str.lower()
ETF_Core_data['leverage_factor'] = np.select(
    [lt.str.contains(r'(?<!\w)3x\b|\btriple\b|ultrapro', regex=True),
     lt.str.contains(r'(?<!\w)2x\b|\bdouble\b|\bultra\b(?![- ]?short)', regex=True)],
    [3, 2], default=1)
ETF_Core_data['leveraged'] = ETF_Core_data['leverage_factor'] > 1

# PATCH: robo-adviser policy, no 2x/3x funds in recommendations
lev = ETF_Core_data['eligible'] & ETF_Core_data['leveraged']
ETF_Core_data.loc[lev, 'exclusion_reason'] = 'leveraged fund (robo-adviser policy)'
ETF_Core_data['eligible'] = ETF_Core_data['exclusion_reason'].isna()
print(lev.sum(), "leveraged funds excluded")

# 3) Checks on the funds that will go into the core database
elig = ETF_Core_data[ETF_Core_data['eligible']]
print(f"{len(elig)} eligible of {len(ETF_Core_data)} funds")
print(elig['listing_country'].value_counts().to_string())
print("\nMER (percent) summary, expect a median around 0.1-0.5:")
print(elig['mer_pct'].describe().round(3).to_string())

problems = {'duplicate tickers': int(ETF_Core_data['ticker'].duplicated().sum())}
for col in ['name', 'ticker_yahoo', 'asset_class', 'etf_category', 'region', 'currency',
            'mer_pct', 'aum_cad_m', 'avg_volume_30d', 'volatility', 'risk_tier']:
    problems[f'eligible with no {col}'] = int(elig[col].isna().sum())
problems['eligible with region Unknown'] = int((elig['region'] == 'Unknown').sum())
problems['eligible in category Other'] = int((elig['etf_category'] == 'Other').sum())
problems['volatility outside 0-100%'] = int((~elig['volatility'].dropna().between(0, 1)).sum())
problems['mer_pct outside 0-3%'] = int((~elig['mer_pct'].dropna().between(0, 3)).sum())
display(pd.Series(problems, name='count').to_frame())

In [ ]:
# PATCH: plain, factual "About this fund" text where Yahoo has no description (all Canadian listings)
def describe(r):
    region = {'Global': 'a global', 'International': 'an international (developed markets outside North America)',
              'Emerging Markets': 'an emerging-markets', 'North America': 'a North American',
              'Canada': 'a Canadian', 'US': 'a US'}.get(r['region'], 'a broad')
    parts = [f"{r['name']} is an exchange-traded fund" + (f" from {r['FundFamily']}" if pd.notna(r['FundFamily']) else '')
             + f", listed in {r['listing_country']} and traded in {r['currency']}."]
    parts.append(f"In the Robo-Kit it sits in the {r['etf_category']} category with {region} focus"
                 + (f" and a {r['theme']} theme." if pd.notna(r['theme']) and r['theme'] != 'ESG' else '.'))
    if pd.notna(r['mer_pct']):
        parts.append(f"It charges a management expense ratio of {r['mer_pct']:.2f}% a year"
                     + (f" and holds about C${r['aum_cad_m']:,.0f} million in assets." if pd.notna(r['aum_cad_m']) else '.'))
    if pd.notna(r['volatility']):
        parts.append(f"Over about three years of weekly prices its annualized volatility was {r['volatility']:.1%}, "
                     f"which places it in the {r['risk_tier']} risk tier.")
    if r['cad_hedged']:
        parts.append('Its foreign-currency exposure is hedged back to Canadian dollars.')
    if r['is_esg']:
        parts.append('It follows a responsible-investing (ESG) mandate.')
    if r['income']:
        parts.append('It focuses on paying regular income.')
    return ' '.join(parts)

has = ETF_Core_data['longBusinessSummary'].fillna('').str.strip() != ''
ETF_Core_data['description_source'] = np.where(has, 'yahoo', None)
gen = ETF_Core_data['eligible'] & ~has
ETF_Core_data.loc[gen, 'longBusinessSummary'] = ETF_Core_data[gen].apply(describe, axis=1)
ETF_Core_data.loc[gen, 'description_source'] = 'generated from fund data'
print(gen.sum(), "descriptions generated")


Build ETF_Core.db and ETF_context.db

In [ ]:
SCHEMA_VERSION = 1
BUILD_NOTE = 'initial build'
BUILD_TS = datetime.now().isoformat(timespec='seconds')

RENAME = {'nameLong': 'name_long', 'FundFamily': 'fund_family', 'ETF_confirmed': 'etf_confirmed',
          'volatility_beta': 'beta', 'volatility_sharpe_ratio': 'sharpe_ratio',
          'risk_maximum_drawdown': 'max_drawdown', 'volatility_52_week_high': 'week52_high',
          'volatility_52_week_low': 'week52_low', 'threeYearAverageReturn': 'three_year_avg_return',
          'theme_ESG_score': 'theme_esg_score', 'risk_holdings_count': 'holdings_count',
          'longBusinessSummary': 'long_business_summary'}

universe = (ETF_Core_data.rename(columns=RENAME)
            .drop(columns=['MER', 'trailing_div_yield'], errors='ignore').copy())
if 'morningstar_category' not in universe.columns:
    universe['morningstar_category'] = None
universe['updated_at'] = BUILD_TS
for c in ['is_esg', 'income', 'leveraged', 'cad_hedged', 'is_inverse', 'etf_confirmed', 'eligible']:
    universe[c] = universe[c].astype('boolean').astype('Int64')
for c in ['price_weeks', 'leverage_factor']:
    universe[c] = universe[c].astype('Int64')

# ---------- table definitions ----------
CORE_ETF = {
    'ticker': 'TEXT PRIMARY KEY', 'ticker_yahoo': 'TEXT NOT NULL', 'name': 'TEXT NOT NULL',
    'fund_family': 'TEXT', 'exchange': 'TEXT', 'market': 'TEXT', 'listing_country': 'TEXT NOT NULL',
    'currency': 'TEXT', 'asset_class': 'TEXT NOT NULL', 'etf_category': 'TEXT NOT NULL',
    'region': 'TEXT NOT NULL', 'theme': 'TEXT',
    'is_esg': 'INTEGER NOT NULL CHECK (is_esg IN (0,1))',
    'income': 'INTEGER CHECK (income IN (0,1))',
    'leveraged': 'INTEGER NOT NULL CHECK (leveraged IN (0,1))',
    'leverage_factor': 'INTEGER', 'cad_hedged': 'INTEGER',
    'mer_pct': 'REAL', 'aum_cad_m': 'REAL', 'avg_volume_30d': 'REAL', 'volatility': 'REAL',
    'risk_tier': "TEXT CHECK (risk_tier IN ('Low','Medium','High'))",
    'updated_at': 'TEXT NOT NULL'}
UNIVERSE = {
    'ticker': 'TEXT PRIMARY KEY', 'ticker_yahoo': 'TEXT', 'ticker_yahoo_initial': 'TEXT', 'verify_status': 'TEXT',
    'name': 'TEXT', 'name_long': 'TEXT', 'fund_family': 'TEXT', 'exchange': 'TEXT', 'market': 'TEXT',
    'listing_country': 'TEXT', 'currency': 'TEXT', 'asset_class': 'TEXT', 'etf_category': 'TEXT',
    'region': 'TEXT', 'region_source': 'TEXT', 'theme': 'TEXT',
    'is_esg': 'INTEGER', 'income': 'INTEGER', 'leveraged': 'INTEGER', 'leverage_factor': 'INTEGER',
    'cad_hedged': 'INTEGER', 'is_inverse': 'INTEGER', 'etf_confirmed': 'INTEGER',
    'mer_pct': 'REAL', 'aum_total_assets': 'REAL', 'aum_cad_m': 'REAL', 'avg_volume_30d': 'REAL',
    'volatility': 'REAL', 'risk_tier': 'TEXT', 'price_weeks': 'INTEGER',
    'eligible': 'INTEGER', 'exclusion_reason': 'TEXT', 'updated_at': 'TEXT'}
PROFILE = {'ticker': 'TEXT PRIMARY KEY', 'name_long': 'TEXT', 'long_business_summary': 'TEXT',
           'morningstar_category': 'TEXT', 'description_source': 'TEXT', 'updated_at': 'TEXT'}   # PATCH
METRICS = {'ticker': 'TEXT PRIMARY KEY', 'max_drawdown': 'REAL', 'sharpe_ratio': 'REAL', 'beta': 'REAL',
           'week52_high': 'REAL', 'week52_low': 'REAL', 'three_year_avg_return': 'REAL',
           'income_yield': 'REAL', 'income_yield_trailing': 'REAL', 'holdings_count': 'REAL',
           'theme_esg_score': 'REAL', 'updated_at': 'TEXT'}

missing = (set(CORE_ETF) | set(UNIVERSE) | set(PROFILE) | set(METRICS)) - set(universe.columns)
assert not missing, f"Columns missing from the DataFrame: {missing}"

CATEGORIES = [('Canadian stocks', 1, 'core'), ('US stocks', 2, 'core'), ('International stocks', 3, 'core'),
              ('Emerging markets stocks', 4, 'core'), ('Canadian bonds (all maturities)', 5, 'core'),
              ('Canadian short-term bonds', 6, 'core'), ('Global bonds (incl. US)', 7, 'core'),
              ('Real estate', 8, 'other'), ('Commodities', 9, 'other'), ('Cash', 10, 'other'),
              ('Sector or theme', 11, 'other'), ('Other', 12, 'other')]

DICT = [
 ('etf', 'ticker', 'Bloomberg-style ticker; primary key in all three databases', 'e.g. VFV CN'),
 ('etf', 'ticker_yahoo', 'Verified Yahoo Finance symbol (can differ from the Excel exchange, e.g. VBU.NE)', 'text'),
 ('etf', 'name', 'Fund name from the source Excel list', 'text'),
 ('etf', 'fund_family', 'Issuer, from Yahoo', 'text'),
 ('etf', 'exchange', 'Exchange code reported by Yahoo', 'e.g. TOR, NEO'),
 ('etf', 'market', 'Venue implied by the Bloomberg code in the Excel; can differ from exchange', 'text'),
 ('etf', 'listing_country', 'Country of the verified listing', 'Canada | US'),
 ('etf', 'currency', 'Trading currency', 'CAD | USD'),
 ('etf', 'asset_class', 'Asset class from the Excel sheet', 'Equity | Bond | Commodity | Real Estate'),
 ('etf', 'etf_category', 'Robo-Kit category: seven core classes plus real estate, commodities, cash, sector or theme, other', 'see etf_category_lookup'),
 ('etf', 'region', 'Geographic focus from fund name, Yahoo category and listing', 'Canada | US | International | Emerging Markets | Global | North America | Unknown'),
 ('etf', 'theme', 'Sector or theme from name keywords; null for broad funds', 'e.g. Technology, Clean Energy, ESG'),
 ('etf', 'is_esg', '1 if the name or category signals an ESG or sustainable mandate', '0 | 1'),
 ('etf', 'income', '1 if dividend-focused (name keywords; stock-type funds only)', '0 | 1'),
 ('etf', 'leveraged', '1 if leveraged (2x or 3x)', '0 | 1'),
 ('etf', 'leverage_factor', 'Leverage multiple parsed from the name', '1 | 2 | 3'),
 ('etf', 'cad_hedged', '1 if the name says the fund is hedged', '0 | 1'),
 ('etf', 'mer_pct', 'Management expense ratio', 'percent: 0.09 means 0.09%'),
 ('etf', 'aum_cad_m', 'Fund size: Yahoo total assets converted to CAD', 'CAD millions'),
 ('etf', 'avg_volume_30d', 'Average daily volume from Yahoo (a 3-month average)', 'units per day'),
 ('etf', 'volatility', 'Annualized volatility from about 3 years of weekly returns', 'fraction: 0.11 = 11%'),
 ('etf', 'risk_tier', 'Low under 11%, Medium 11-20%, High 20% and over', 'Low | Medium | High'),
 ('etf', 'updated_at', 'When the row was last written', 'ISO timestamp'),
 ('etf_metrics', 'max_drawdown', 'Worst peak-to-trough fall in weekly closes (about 3 years)', 'fraction, negative'),
 ('etf_metrics', 'sharpe_ratio', 'Annualized mean weekly return divided by volatility; risk-free rate 0', 'ratio'),
 ('etf_metrics', 'beta', 'Yahoo 3-year beta where available', 'ratio'),
 ('etf_metrics', 'income_yield / income_yield_trailing', 'Yahoo yield and trailing distribution yield', 'check units against Yahoo'),
 ('etf_universe', 'eligible', '1 if the fund passed every exclusion rule and is in etf_core.db', '0 | 1'),
 ('etf_universe', 'exclusion_reason', 'First rule that removed the fund; null if eligible', 'text'),
 # PATCH
 ('etf_profile', 'description_source', 'Where long_business_summary came from', 'yahoo | generated from fund data'),
 ('etf', 'asset_class', "Adds 'Crypto' for bitcoin funds (Additions sheet), capped through allocation_policy", 'Equity | Bond | Commodity | Real Estate | Crypto'),
 ('allocation_policy', 'max_weight', 'Largest total portfolio weight for the asset class', 'fraction: 0.05 = 5%')]

ALLOCATION_POLICY = [   # PATCH: (asset_class, allowed_risk_profiles, max_weight, note)
    ('Crypto', 'High', 0.05, 'Offer only to High-risk clients as an add-on; cap the whole crypto sleeve at 5%')]

EXCLUSION_RULES = ('Canada/US listings only; confirmed ETFs; no inverse funds; no missing key figures; '
                   'size >= C$50M (US$250M for US-listed); volume >= 10,000/day (1,000 for Canadian-listed); '
                   '>= 150 weekly prices; no leveraged funds')   # PATCH

# ---------- helpers ----------
def make_table(con, name, cols):
    con.execute(f"CREATE TABLE IF NOT EXISTS {name} (" + ', '.join(f'{c} {t}' for c, t in cols.items()) + ")")

def rows(d):
    return [tuple(None if pd.isna(v) else (v.item() if hasattr(v, 'item') else v) for v in r)
            for r in d.itertuples(index=False, name=None)]

def upsert(con, table, d, key='ticker'):
    cols = list(d.columns)
    sets = ', '.join(f'{c}=excluded.{c}' for c in cols if c != key)
    con.executemany(f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))}) "
                    f"ON CONFLICT({key}) DO UPDATE SET {sets}", rows(d))

def set_meta(con, items):
    con.executemany("INSERT INTO meta VALUES (?,?) ON CONFLICT(key) DO UPDATE SET value=excluded.value",
                    [(k, str(v)) for k, v in items.items()])

core_df = universe.loc[universe['eligible'] == 1, list(CORE_ETF)]

# ---------- etf_core.db ----------
con = sqlite3.connect('etf_core.db')
with con:
    make_table(con, 'etf', CORE_ETF)
    make_table(con, 'etf_category_lookup', {'etf_category': 'TEXT PRIMARY KEY', 'sort_order': 'INTEGER', 'class_group': 'TEXT'})
    make_table(con, 'meta', {'key': 'TEXT PRIMARY KEY', 'value': 'TEXT'})
    con.execute("CREATE INDEX IF NOT EXISTS ix_etf_cat_risk ON etf(etf_category, risk_tier)")
    con.execute("CREATE INDEX IF NOT EXISTS ix_etf_region ON etf(region)")
    make_table(con, 'allocation_policy', {'asset_class': 'TEXT PRIMARY KEY', 'allowed_risk_profiles': 'TEXT',   # PATCH
                                          'max_weight': 'REAL', 'note': 'TEXT'})
    con.executemany("INSERT OR REPLACE INTO allocation_policy VALUES (?,?,?,?)", ALLOCATION_POLICY)
    upsert(con, 'etf', core_df)
    con.execute("CREATE TEMP TABLE keep(ticker TEXT PRIMARY KEY)")
    con.executemany("INSERT INTO keep VALUES (?)", [(t,) for t in core_df['ticker']])
    removed = con.execute("DELETE FROM etf WHERE ticker NOT IN (SELECT ticker FROM keep)").rowcount
    con.executemany("INSERT INTO etf_category_lookup VALUES (?,?,?) ON CONFLICT(etf_category) DO UPDATE "
                    "SET sort_order=excluded.sort_order, class_group=excluded.class_group", CATEGORIES)
    set_meta(con, {'schema_version': SCHEMA_VERSION, 'built_at': BUILD_TS, 'rows': len(core_df),
                   'source': 'ETF List Data.xlsx + Yahoo Finance (yfinance)', 'exclusion_rules': EXCLUSION_RULES})
con.execute(f"PRAGMA user_version = {SCHEMA_VERSION}")
con.commit(); con.close()
_px = sqlite3.connect('etf_prices.db'); _px.execute(f"PRAGMA user_version = {SCHEMA_VERSION}"); _px.close()   # PATCH
print(f"etf_core.db: {len(core_df)} funds written, {removed} no longer eligible removed")

# ---------- etf_context.db ----------
tickers = set(universe['ticker'])
raw_rows = [(t, c.get('resolved'), json.dumps(c['data'], default=str))
            for t, c in cache.items() if c.get('data') and t in tickers]

con = sqlite3.connect('etf_context.db')
with con:
    make_table(con, 'etf_universe', UNIVERSE)
    make_table(con, 'etf_profile', PROFILE)
    if 'description_source' not in [r[1] for r in con.execute('PRAGMA table_info(etf_profile)')]:   # PATCH
        con.execute('ALTER TABLE etf_profile ADD COLUMN description_source TEXT')
    make_table(con, 'etf_metrics', METRICS)
    make_table(con, 'yahoo_raw', {'ticker': 'TEXT PRIMARY KEY', 'ticker_yahoo': 'TEXT', 'raw_json': 'TEXT'})
    make_table(con, 'data_dictionary', {'table_name': 'TEXT', 'column_name': 'TEXT', 'definition': 'TEXT', 'units_or_values': 'TEXT'})
    make_table(con, 'refresh_log', {'run_at': 'TEXT', 'step': 'TEXT', 'rows_core': 'INTEGER', 'rows_universe': 'INTEGER', 'note': 'TEXT'})
    make_table(con, 'meta', {'key': 'TEXT PRIMARY KEY', 'value': 'TEXT'})
    con.execute("CREATE VIEW IF NOT EXISTS excluded_etfs AS SELECT ticker, name, listing_country, exclusion_reason "
                "FROM etf_universe WHERE eligible = 0")
    upsert(con, 'etf_universe', universe[list(UNIVERSE)])
    upsert(con, 'etf_profile', universe[list(PROFILE)])
    upsert(con, 'etf_metrics', universe[list(METRICS)])
    con.executemany("INSERT INTO yahoo_raw VALUES (?,?,?) ON CONFLICT(ticker) DO UPDATE "
                    "SET ticker_yahoo=excluded.ticker_yahoo, raw_json=excluded.raw_json", raw_rows)
    con.execute("DELETE FROM data_dictionary")
    con.executemany("INSERT INTO data_dictionary VALUES (?,?,?,?)", DICT)
    con.execute("INSERT INTO refresh_log VALUES (?,?,?,?,?)",
                (BUILD_TS, 'full build', len(core_df), len(universe), BUILD_NOTE))
    set_meta(con, {'schema_version': SCHEMA_VERSION, 'built_at': BUILD_TS, 'rows_universe': len(universe),
                   'exclusion_rules': EXCLUSION_RULES})
con.execute(f"PRAGMA user_version = {SCHEMA_VERSION}")
con.commit()
if os.path.exists('etfs_not_found_on_yahoo.csv'):      # funds dropped earlier, kept as a record
    pd.read_csv('etfs_not_found_on_yahoo.csv').to_sql('not_found_on_yahoo', con, if_exists='replace', index=False)
con.close()
print(f"etf_context.db: {len(universe)} funds in universe, {len(raw_rows)} raw Yahoo records")

verify and make handover copies

In [28]:
con = sqlite3.connect('etf_core.db')
con.execute("ATTACH DATABASE 'etf_context.db' AS ctx")
con.execute("ATTACH DATABASE 'etf_prices.db' AS px")

display(pd.read_sql("""
    SELECT l.sort_order, e.etf_category, COUNT(*) AS funds,
           SUM(risk_tier='Low') AS low, SUM(risk_tier='Medium') AS medium, SUM(risk_tier='High') AS high
    FROM etf e JOIN etf_category_lookup l USING (etf_category)
    GROUP BY e.etf_category ORDER BY l.sort_order""", con))

checks = {
    'core funds missing from the universe table': "SELECT COUNT(*) FROM etf e LEFT JOIN ctx.etf_universe u USING (ticker) WHERE u.ticker IS NULL",
    'core funds missing a profile':               "SELECT COUNT(*) FROM etf e LEFT JOIN ctx.etf_profile p USING (ticker) WHERE p.ticker IS NULL",
    'core funds without weekly prices':           "SELECT COUNT(*) FROM etf e LEFT JOIN px.price_meta m ON m.ticker = e.ticker AND m.status = 'ok' WHERE m.ticker IS NULL",
    'core funds without a risk tier':             "SELECT COUNT(*) FROM etf WHERE risk_tier IS NULL",
}
for name, sql in checks.items():
    print(f"{name:45s} {con.execute(sql).fetchone()[0]}")      # all should be 0
print("integrity:", con.execute("PRAGMA integrity_check").fetchone()[0])
con.close()

# Stamp the prices file like the others, then make dated copies for the team
pc = sqlite3.connect('etf_prices.db')
pc.execute("CREATE TABLE IF NOT EXISTS meta (key TEXT PRIMARY KEY, value TEXT)")
pc.executemany("INSERT INTO meta VALUES (?,?) ON CONFLICT(key) DO UPDATE SET value=excluded.value",
               [('schema_version', str(SCHEMA_VERSION)), ('built_at', BUILD_TS),
                ('frequency', 'weekly, week ending Friday, dividend-adjusted close')])
pc.commit(); pc.close()

os.makedirs('handover', exist_ok=True)
for f in ['etf_core.db', 'etf_context.db', 'etf_prices.db']:
    shutil.copy(f, f"handover/{f[:-3]}_v{SCHEMA_VERSION}_{datetime.now():%Y%m%d}.db")
print("copies in ./handover")

,sort_order,etf_category,funds,low,medium,high
0,1,Canadian stocks,7,0,7,0
1,2,US stocks,65,6,53,6
2,3,International stocks,197,7,184,6
3,4,Emerging markets stocks,88,0,65,23
4,5,Canadian bonds (all maturities),14,14,0,0
5,7,Global bonds (incl. US),65,64,1,0
6,8,Real estate,30,0,28,2
7,9,Commodities,48,1,29,18
8,10,Cash,6,6,0,0
9,11,Sector or theme,8,4,1,3


core funds missing from the universe table    0
core funds missing a profile                  0
core funds without weekly prices              0
core funds without a risk tier                0
integrity: ok
copies in ./handover
